In [1]:
# --- Portable bootstrap: locate scripts/ from any working directory ---
import sys
from pathlib import Path

def _find_scripts_dir():
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        for cand in (base / "scripts", base,
                     *sorted(base.glob("*/scripts")), *sorted(base.glob("*/*/scripts"))):
            if (cand / "project_paths.py").is_file():
                return cand
    raise FileNotFoundError(
        "scripts/project_paths.py not found. Open this notebook from inside the "
        "cloned repository, or point the kernel's working directory at it."
    )

sys.path.insert(0, str(_find_scripts_dir()))
from project_config import *   # SEED, DATA_DIR, ACOUSTIC_FULL, collapse_functionals, repeated_cv, ...

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

# --- Data ---
model_df = load_model_df()
y = model_df["phq9"]
acoustic = ACOUSTIC_FULL            # the 88 raw eGeMAPS features these cells start from

# --- Full-sample correlation structure, for the DENDROGRAM and THRESHOLD-SWEEP
# figures only. Descriptive, never scored -- the modelling cells refit the
# clustering per fold via CorrelationClusterSelector.
_feat = pd.read_csv(EGEMAPS_CSV, dtype={"subject_id": str}).dropna(subset=["phq9"])
corr = np.nan_to_num(spearmanr(_feat[acoustic]).correlation, nan=0.0)
corr = (corr + corr.T) / 2
np.fill_diagonal(corr, 1.0)
Z = linkage(squareform(1.0 - np.abs(corr), checks=False), method="average")


In [14]:
# ===== Nested CV: unbiased estimate of the SELECTED acoustic reduction pipeline =====
#
# Outer 5-fold: outer_test is never touched during selection.
#   Inner (outer_train only): repeated 10-fold CV over a GRID of (strategy,
#   hyperparameter) configs; the winner is chosen on outer_train alone, refit on
#   outer_train, and scored on the untouched outer_test.
#
# Hyperparameters are NOT fixed from full-sample inspection. The clustering
# threshold and the PCA variance target are searched inside the inner loop, so
# each outer fold picks its own. The GRID itself is an a-priori search space, not
# a tuned value. The selection metric is also not a free choice: the whole
# procedure is run under both MAE and R2 and both outer estimates are reported.
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# --- Config -----------------------------------------------------------------
N_OUTER     = 5
INNER_SEEDS = range(5)          # repeated 10-fold CV inside each outer fold
N_JOBS      = -1
SELECTION_METRICS = ("MAE", "R2")
HIGHER_IS_BETTER  = {"R2": True, "r": True, "MAE": False, "RMSE": False}

# Search space, fixed a priori: spans the plausible range, not tuned on results
GRID = {
    "A-priori collapse": [{}],                                            # no hyperparameter
    "Corr. clustering":  [{"threshold": t} for t in (0.60, 0.70, 0.75, 0.80, 0.90)],
    "PCA":               [{"pca_var":  v} for v in (0.80, 0.90, 0.95)],
}

X_full       = model_df[ACOUSTIC_FULL]                 # all 88 eGeMAPS features
apriori_cols = collapse_functionals(ACOUSTIC_FULL)     # -> 72, by name only
apriori_idx  = [X_full.columns.get_loc(c) for c in apriori_cols]


class ColumnSubset(BaseEstimator, TransformerMixin):
    """Fixed column subset. No fitting -> cannot leak."""
    def __init__(self, idx): self.idx = idx
    def fit(self, X, y=None): return self
    def transform(self, X): return np.asarray(X, dtype=float)[:, self.idx]


class CorrelationClusterSelector(BaseEstimator, TransformerMixin):
    """Spearman |rho| -> average-linkage clustering -> one medoid per cluster.
    Never sees y, but is data-dependent, so fit on training rows only."""
    def __init__(self, threshold=0.75): self.threshold = threshold
    def fit(self, X, y=None):
        corr = np.nan_to_num(spearmanr(np.asarray(X, float)).correlation, nan=0.0)
        corr = (corr + corr.T) / 2
        np.fill_diagonal(corr, 1.0)
        cl = fcluster(linkage(squareform(1 - np.abs(corr), checks=False), "average"),
                      t=1 - self.threshold, criterion="distance")
        absc = np.abs(corr)
        self.support_ = np.sort([i[np.argmax(absc[np.ix_(i, i)].sum(1))]
                                 for i in (np.where(cl == c)[0] for c in np.unique(cl))])
        return self
    def transform(self, X): return np.asarray(X, float)[:, self.support_]


def make_pipe(family, params):
    """Fresh pipeline for one grid point."""
    rf = RandomForestRegressor(random_state=SEED)
    if family == "A-priori collapse":
        return Pipeline([("reduce", ColumnSubset(apriori_idx)), ("rf", rf)])
    if family == "Corr. clustering":
        return Pipeline([("reduce", CorrelationClusterSelector(params["threshold"])), ("rf", rf)])
    return Pipeline([("scale",  StandardScaler()),
                     ("reduce", PCA(n_components=params["pca_var"], random_state=SEED)),
                     ("rf",     rf)])

def label(family, params):
    return family if not params else f"{family} ({next(iter(params.values()))})"

def n_kept(pipe):
    step = pipe.named_steps["reduce"]
    if isinstance(step, ColumnSubset):               return len(step.idx)
    if isinstance(step, CorrelationClusterSelector): return len(step.support_)
    return step.n_components_                        # PCA

def inner_cv(est, X, y, seeds=INNER_SEEDS, n_splits=10):
    rows = []
    for s in seeds:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=s)
        p = cross_val_predict(est, X, y, cv=kf, n_jobs=N_JOBS)
        rows.append({"MAE": mean_absolute_error(y, p),
                     "RMSE": np.sqrt(mean_squared_error(y, p)),
                     "R2": r2_score(y, p)})
    d = pd.DataFrame(rows)
    return {k: (d[k].mean(), d[k].std()) for k in ["MAE", "RMSE", "R2"]}


CONFIGS = [(fam, p) for fam, plist in GRID.items() for p in plist]
print(f"Grid: {len(CONFIGS)} configs x {N_OUTER} outer folds x {len(INNER_SEEDS)} inner seeds")

outer = KFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
oof        = {m: np.full(len(y), np.nan) for m in SELECTION_METRICS}
fold_rows, inner_rows = [], []

for f, (tr, te) in enumerate(outer.split(X_full), start=1):
    X_tr, X_te = X_full.iloc[tr], X_full.iloc[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]

    # --- inner: score every grid point on outer_train ONLY (computed once) ---
    scores = {}
    for i, (fam, params) in enumerate(CONFIGS):
        scores[i] = inner_cv(make_pipe(fam, params), X_tr, y_tr)
        inner_rows.append({"fold": f, "config": label(fam, params), "family": fam,
                           "inner_MAE": scores[i]["MAE"][0], "inner_R2": scores[i]["R2"][0]})

    # --- select + evaluate under each metric, from the same inner scores ---
    for metric in SELECTION_METRICS:
        pick = max if HIGHER_IS_BETTER[metric] else min
        i    = pick(scores, key=lambda k: scores[k][metric][0])
        fam, params = CONFIGS[i]

        best = make_pipe(fam, params)
        best.fit(X_tr, y_tr)
        pred = best.predict(X_te)
        oof[metric][te] = pred

        fold_rows.append({"fold": f, "select_on": metric, "winner": label(fam, params),
                          "family": fam, "n_features": n_kept(best), "n_test": len(te),
                          "outer_MAE": mean_absolute_error(y_te, pred),
                          "outer_R2":  r2_score(y_te, pred)})
        print(f"fold {f} [select on {metric:3s}]: {label(fam, params):26s} "
              f"({n_kept(best):>2d} feats)  outer MAE {fold_rows[-1]['outer_MAE']:.3f}")

folds = pd.DataFrame(fold_rows)
inner = pd.DataFrame(inner_rows)

print("\n=== Inner scores per config (mean R2 across outer folds) ===")
print(inner.groupby("config")[["inner_R2", "inner_MAE"]].mean().round(3).to_string())

print("\n=== Outer (unbiased) performance ===")
for metric in SELECTION_METRICS:
    sub, p = folds[folds["select_on"] == metric], oof[metric]
    print(f"\n  selected on {metric}:")
    print(f"    winners            : {sub['winner'].tolist()}")
    print(f"    MAE averaged/folds : {sub['outer_MAE'].mean():.3f} ± {sub['outer_MAE'].std():.3f}")
    print(f"    pooled             : MAE {mean_absolute_error(y, p):.3f}   "
          f"R2 {r2_score(y, p):.3f}   RMSE {np.sqrt(mean_squared_error(y, p)):.3f}   "
          f"r {pearsonr(y, p)[0]:.3f}")

folds.to_csv(CSV_DIR / "nested_cv_folds.csv", index=False)
inner.to_csv(CSV_DIR / "nested_cv_inner_scores.csv", index=False)


Grid: 9 configs x 5 outer folds x 5 inner seeds
fold 1 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.461
fold 1 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.461
fold 2 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.363
fold 2 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.363
fold 3 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 5.645
fold 3 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 5.645
fold 4 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.404
fold 4 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.404
fold 5 [select on MAE]: A-priori collapse          (72 feats)  outer MAE 6.311
fold 5 [select on R2 ]: A-priori collapse          (72 feats)  outer MAE 6.311

=== Inner scores per config (mean R2 across outer folds) ===
                         inner_R2  inner_MAE
config                                      
A-priori c

In [13]:
print(folds[["fold", "n_test"]])

   fold  n_test
0     1      11
1     2      11
2     3      10
3     4      10
4     5      10
